Phi-4-mini-instruct | PEV at a single fixed sigma
All 100 JBB-Behaviors prompts, 20 generations per prompt, run as ONE batch of 20.

Differences from the sweep notebook:
  - PERTURB is a single fixed value (0.08, the Phi-4-mini peak), not a list
  - all 100 prompts in one notebook (no 4-way 25-prompt split)
  - the 20 runs per prompt are generated in a single batched pass with independent
    noise per row, instead of 20 sequential run_one() calls

The verification cells (qkv split, cache consistency, smoke test) are unchanged, and
run_one() is kept because the smoke test uses it. Log format is unchanged, so the
existing categorization pipeline parses it as-is.

In [ ]:
get_ipython().system('pip -q install -U transformers accelerate datasets huggingface_hub')

print("Now: Runtime > Restart session, then run the next cell.")

In [ ]:
# Run AFTER restarting the session.
import torch
print("torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM: {total/1e9:.1f} GB total, {free/1e9:.1f} GB free")
    print("Phi-4-mini needs ~7.6 GB in bf16, plus KV cache for the batch of 20.")

import transformers
print("transformers:", transformers.__version__)

In [ ]:
from google.colab import files
uploaded = files.upload()  # select run_phi4mini.py from your machine

### HF login

`microsoft/Phi-4-mini-instruct` is MIT and **not gated**. Optional.

In [ ]:
from huggingface_hub import login
from getpass import getpass
import os

hf_token = getpass("Enter your Hugging Face token (hf_..., or blank to skip): ").strip()
if hf_token:
    login(token=hf_token)
    os.environ["HF_TOKEN"] = hf_token
    print("HF login successful")
else:
    print("Skipping login (repo is public and ungated)")

In [ ]:
import os
os.environ["PHI4_DEVICE"] = "cuda"
os.environ["PHI4_SKIP_EMBED_PROMPT"] = "0"
# os.environ["PHI4_TRUST_REMOTE_CODE"] = "1"   # force the remote-code path if needed

import torch
import run_phi4mini as rp

model, tokenizer = rp.load_model()
model.eval()

DEVICE = getattr(rp, "DEVICE", None) or next(model.parameters()).device
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
print("device:", DEVICE, "| dtype:", next(model.parameters()).dtype)

### Verify the fused qkv split, the load-bearing check for this model

Confirms the fused width equals `(n_q + 2*n_kv) * head_dim` and that the query block is
the leading slice. If this is wrong, every query vector in the analysis is wrong, and
nothing else in the pipeline would tell you.

In [ ]:
info = rp.verify_qkv_split(model)
print(info)
assert info["n_layers"] == model.config.num_hidden_layers, "hooks would miss layers"
print("\nQuery-vector capture verified.")

In [ ]:
stats = rp.embedding_stats(model)
for s in [0.005, 0.008, 0.012, 0.016, 0.02, 0.024, 0.03, 0.04, 0.05, 0.06, 0.08, 0.1]:
    print(f"sigma={s:<7} relative to row RMS: {s / stats['row_rms_mean']:.4f}")

### Cache consistency check

In [ ]:
ok = rp.verify_cache_consistency(model, tokenizer, n_tokens=24)
assert ok, "Manual loop diverged from generate(). Do not run the sweep until this passes."
print("Cache handling verified.")

In [ ]:
base_config = {
    "verbose": False,
    "temperature": 0.7,
    "top_k": 50,
    "top_p": 0.9,
    "max_new_tokens": 4096,
    "print_pre_layer_embed": False,
    "write_phi4_embed_log_file": False,
    "require_eos_before_print": False,
}

In [ ]:
from datasets import load_dataset

ds = load_dataset("JailbreakBench/JBB-Behaviors", "behaviors", split="harmful")

print("Columns:", ds.column_names)
print("Total rows:", len(ds))

prompt_col = "Goal"
print("Using prompt column:", prompt_col)
print("Sample prompt:")
print(ds[0][prompt_col])

In [ ]:
import os
import math
import torch

def to_safe_text(x):
    if x is None:
        return ""
    if isinstance(x, float) and math.isnan(x):
        return ""
    return str(x)


def run_one(prompt_text, perturb: float):
    """Single sequential generation. Kept for the smoke test below."""
    prompt_text = to_safe_text(prompt_text).strip()

    original_modify = rp.ipynb_style_maybe_modify_embeddings
    original_inspect_in = rp.ipynb_style_inspect_input_print
    original_inspect_layers = rp.ipynb_style_inspect_layers_print
    prev_skip = os.environ.get("PHI4_SKIP_EMBED_PROMPT", "")

    def fixed_perturb(embeddings):
        if perturb <= 0:
            return 0.0, None
        noise = perturb * torch.randn_like(embeddings)
        return perturb, embeddings + noise

    def no_op(*args, **kwargs):
        return None

    rp.ipynb_style_maybe_modify_embeddings = fixed_perturb
    rp.ipynb_style_inspect_input_print = no_op
    rp.ipynb_style_inspect_layers_print = no_op
    os.environ["PHI4_SKIP_EMBED_PROMPT"] = "0"

    try:
        messages = [
            {"role": "system", "content": ""},
            {"role": "user", "content": prompt_text},
        ]
        prompt_text_chat = rp.build_prompt(tokenizer, messages)
        answer = rp.generate_response(model, tokenizer, prompt_text_chat, base_config)
        return to_safe_text(answer).strip()
    finally:
        rp.ipynb_style_maybe_modify_embeddings = original_modify
        rp.ipynb_style_inspect_input_print = original_inspect_in
        rp.ipynb_style_inspect_layers_print = original_inspect_layers
        os.environ["PHI4_SKIP_EMBED_PROMPT"] = prev_skip

### Batched generation

Replaces n sequential run_one() calls with one batched pass. Same chat template and
decoding settings; noise is sigma * randn on the embeddings of the whole templated
prompt, drawn independently per row, which matches the run_one hook.

In [ ]:
GEN_KWARGS = dict(
    do_sample=True,
    temperature=base_config["temperature"],
    top_k=base_config["top_k"],
    top_p=base_config["top_p"],
    max_new_tokens=base_config["max_new_tokens"],
)


def _decode_new_tokens(out_ids, input_ids):
    """With inputs_embeds only, HF returns just the new tokens; be defensive anyway."""
    L = input_ids.shape[1]
    start = 0
    if out_ids.shape[1] >= L and torch.equal(out_ids[0, :L].cpu(), input_ids[0].cpu()):
        start = L
    return tokenizer.batch_decode(out_ids[:, start:], skip_special_tokens=True)


@torch.no_grad()
def run_batch(prompt_text, perturb, n_runs, micro_batch=None, seed=None):
    """Return n_runs completions for one prompt, generated in batches of micro_batch."""
    prompt_text = to_safe_text(prompt_text).strip()
    if not prompt_text:
        return [""] * n_runs
    if seed is not None:
        torch.manual_seed(seed)

    messages = [
        {"role": "system", "content": ""},
        {"role": "user", "content": prompt_text},
    ]
    chat = rp.build_prompt(tokenizer, messages)
    enc = tokenizer(chat, return_tensors="pt", add_special_tokens=False)
    input_ids = enc["input_ids"].to(DEVICE)
    attn_1 = enc["attention_mask"].to(DEVICE)

    embed = model.get_input_embeddings()
    base = embed(input_ids)  # (1, L, H)

    micro_batch = micro_batch or n_runs
    outs = []
    done = 0
    while done < n_runs:
        b = min(micro_batch, n_runs - done)
        try:
            inp = base.expand(b, -1, -1).clone()
            if perturb > 0:
                inp = inp + perturb * torch.randn_like(inp)  # independent noise per row
            out_ids = model.generate(
                inputs_embeds=inp,
                attention_mask=attn_1.expand(b, -1),
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                **GEN_KWARGS,
            )
            outs.extend(_decode_new_tokens(out_ids, input_ids))
            done += b
            del inp, out_ids
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if micro_batch == 1:
                raise
            micro_batch = max(1, micro_batch // 2)
            print(f"    [OOM] retrying with micro_batch={micro_batch}")
    return [to_safe_text(o).replace("\r\n", "\n").strip() for o in outs]

### Smoke test

Confirms the query hooks fire during a real generation, and that the batched path
produces sane output at the sigma we are about to sweep.

In [ ]:
probe = ds[75][prompt_col]
pt = rp.build_prompt(tokenizer, [
    {"role": "system", "content": ""},
    {"role": "user", "content": probe},
])
print("PROMPT TEXT SENT TO MODEL:")
print(repr(pt))
print("prompt tokens:", rp.encode_prompt(tokenizer, pt).shape[1])
print()

out = run_one(probe, 0.024)
print(f"--- run_one sigma=0.024 | chars={len(out)} ---")
print(out[:400])
print()

caps = rp._captured_query_vectors
assert caps, "query hooks did not fire during generation"
print(f"query hooks fired on {len(caps)} layers | q width = "
      f"{list(caps.values())[0].shape[-1]} (expected {info['q_width']})")
print()

batch_probe = run_batch(probe, 0.08, 3, seed=0)
for i, o in enumerate(batch_probe, 1):
    print(f"--- run_batch sigma=0.08 row {i} | chars={len(o)} ---")
    print(o[:300])
    print()

In [ ]:
import os
import time
from datetime import datetime

# Mount Drive so logs survive Colab disconnect (set False only for a quick test)
USE_DRIVE = True
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_DIR = "/content/drive/MyDrive/LLm_perturbation_logs_phi4mini"
else:
    LOG_DIR = "/content"
os.makedirs(LOG_DIR, exist_ok=True)

PERTURB = 0.08             # single fixed sigma (Phi-4-mini peak)
START_PROMPT = 1           # 1-indexed; bump this to resume after a disconnect
NUM_PROMPTS = 100
RUNS_PER_PERTURB = 20
MICRO_BATCH = 20           # all 20 in one pass; auto-halves on OOM
SEED = 1234                # per-prompt seed = SEED + prompt index; None to disable

start_idx = START_PROMPT - 1
end_idx = min(start_idx + NUM_PROMPTS, len(ds))
prompt_indices = list(range(start_idx, end_idx))
prompts = [to_safe_text(ds[i][prompt_col]).strip() for i in prompt_indices]
num_prompts = len(prompts)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
log_path = os.path.join(
    LOG_DIR,
    f"JBB_phi4mini_sigma{str(PERTURB).replace('.', 'p')}"
    f"_p{START_PROMPT:03d}_to_p{start_idx + num_prompts:03d}_{ts}.txt",
)

total = num_prompts * RUNS_PER_PERTURB
print("Log file:", log_path)
print("Prompts:", num_prompts, "| sigma:", PERTURB, "| Runs each:", RUNS_PER_PERTURB)
print("Total generations:", total)

In [ ]:
import shutil

local_log = f"/content/{os.path.basename(log_path)}"


def checkpoint_copy():
    shutil.copy2(log_path, local_log)


job = 0
t0 = time.time()

with open(log_path, "a", encoding="utf-8") as f:
    f.write(f"start {datetime.now().isoformat()}\n")
    f.write(f"model={rp.MODEL_NAME!r} dtype={rp.DTYPE}\n")
    f.write(f"fused_qkv={info['fused']} q_width={info['q_width']} head_dim={info['head_dim']}\n")
    f.write(f"embed_row_rms={stats['row_rms_mean']:.6g}\n")
    f.write(f"prompt_col={prompt_col!r} num_prompts={num_prompts} runs={RUNS_PER_PERTURB}\n")
    f.write(f"perturb_values=[{PERTURB}]\n")
    f.write(f"batched=True micro_batch={MICRO_BATCH} seed={SEED}\n")
    f.write("=" * 120 + "\n\n")
    f.flush()
    os.fsync(f.fileno())

    for p_idx, prompt_text in enumerate(prompts, start=START_PROMPT):
        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"perturbation value- {PERTURB}\n")
        f.flush()
        os.fsync(f.fileno())

        print(f"\n>>> Prompt {p_idx}/{START_PROMPT + num_prompts - 1} | perturbation={PERTURB}")
        tp = time.time()

        try:
            outs = run_batch(
                prompt_text,
                PERTURB,
                RUNS_PER_PERTURB,
                micro_batch=MICRO_BATCH,
                seed=None if SEED is None else SEED + p_idx,
            )
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * RUNS_PER_PERTURB

        for run_id, out in enumerate(outs, start=1):
            job += 1
            f.write(f"run{run_id}- {out}\n")
        f.write("-" * 120 + "\n")
        f.flush()
        os.fsync(f.fileno())

        chars = sum(len(o) for o in outs) // max(1, len(outs))
        elapsed = time.time() - t0
        eta = (elapsed / job) * (total - job) / 60 if job else 0
        print(
            f"    {RUNS_PER_PERTURB} runs in {time.time() - tp:.1f}s"
            f" | global {job}/{total} | mean chars={chars} | ETA {eta:.0f} min"
        )
        checkpoint_copy()

checkpoint_copy()
print(f"\nDone in {(time.time() - t0) / 60:.1f} min | log: {log_path}")
print(f"Local copy: {local_log}")

### Optional: re-run selected prompts with more samples
Same sigma, a chosen subset of prompt ids, N runs each. Writes its own SELECT log.

In [ ]:
PROMPT_IDS = [10, 17, 21, 35, 100]   # 1-indexed, matches P<n> in the logs
SELECT_RUNS = 30
SELECT_SEED = 4321                   # different from the main pass, so these are fresh samples

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
tag = "-".join(str(p) for p in PROMPT_IDS)
sel_log = os.path.join(
    LOG_DIR,
    f"JBB_phi4mini_sigma{str(PERTURB).replace('.', 'p')}_SELECT_p{tag}_n{SELECT_RUNS}_{ts}.txt",
)
sel_local = f"/content/{os.path.basename(sel_log)}"
sel_total = len(PROMPT_IDS) * SELECT_RUNS
print("Log file:", sel_log, "| total generations:", sel_total)

job = 0
t0 = time.time()

with open(sel_log, "a", encoding="utf-8") as f:
    f.write(f"start {datetime.now().isoformat()}\n")
    f.write(f"model={rp.MODEL_NAME!r} dtype={rp.DTYPE}\n")
    f.write(f"prompt_col={prompt_col!r} prompt_ids={PROMPT_IDS} runs={SELECT_RUNS}\n")
    f.write(f"perturb_values=[{PERTURB}]\n")
    f.write(f"batched=True micro_batch={MICRO_BATCH} seed={SELECT_SEED}\n")
    f.write("=" * 120 + "\n\n")
    f.flush()
    os.fsync(f.fileno())

    for p_idx in PROMPT_IDS:
        prompt_text = to_safe_text(ds[p_idx - 1][prompt_col]).strip()

        f.write(f"Prompt{p_idx} - {prompt_text}\n")
        f.write(f"perturbation value- {PERTURB}\n")
        f.flush()
        os.fsync(f.fileno())

        print(f"\n>>> Prompt {p_idx} | perturbation={PERTURB}")
        tp = time.time()

        try:
            outs = run_batch(
                prompt_text,
                PERTURB,
                SELECT_RUNS,
                micro_batch=MICRO_BATCH,
                seed=None if SELECT_SEED is None else SELECT_SEED + p_idx,
            )
        except Exception as e:
            outs = [f"[ERROR] {type(e).__name__}: {e}"] * SELECT_RUNS

        for run_id, out in enumerate(outs, start=1):
            job += 1
            f.write(f"run{run_id}- {out}\n")
        f.write("-" * 120 + "\n")
        f.flush()
        os.fsync(f.fileno())

        print(f"    {SELECT_RUNS} runs in {time.time() - tp:.1f}s | global {job}/{sel_total}")
        shutil.copy2(sel_log, sel_local)

print(f"\nDone in {(time.time() - t0) / 60:.1f} min | log: {sel_log}")